In [1]:
import polars as pl 
import polars.selectors as cs

In [7]:

df_id_train = pl.scan_csv("../Data/Raw/train_identity.csv")
df_tr_train = pl.scan_csv("../Data/Raw/train_transaction.csv")

join_train = df_tr_train.join(df_id_train,on="TransactionID",how="left")
join_train.sink_parquet("../Data/Raw/train.parquet")

df_id_test = pl.scan_csv("../Data/Raw/test_identity.csv")
df_tr_test = pl.scan_csv("../Data/Raw/test_transaction.csv")

join_test = df_tr_test.join(df_id_test,on="TransactionID",how="left")
join_test.sink_parquet("../Data/Raw/Test.parquet")

In [8]:
df_train = pl.scan_parquet("../Data/Raw/train.parquet")
df_test = pl.scan_parquet("../Data/Raw/Test.parquet")
print(df_train.head(5).collect().glimpse())

Rows: 5
Columns: 434
$ TransactionID  <i64> 2987177, 2987183, 2987192, 2987212, 2987216
$ isFraud        <i64> 0, 0, 0, 0, 0
$ TransactionDT  <i64> 89341, 89444, 89600, 89844, 89916
$ TransactionAmt <f64> 306.47, 200.0, 5.982, 117.0, 100.0
$ ProductCD      <str> 'W', 'H', 'C', 'W', 'W'
$ card1          <i64> 16132, 10047, 3154, 2560, 10057
$ card2          <f64> 111.0, 111.0, 408.0, 562.0, 225.0
$ card3          <f64> 150.0, 150.0, 185.0, 150.0, 150.0
$ card4          <str> 'visa', 'visa', 'mastercard', 'visa', 'mastercard'
$ card5          <f64> 226.0, 226.0, 224.0, 162.0, 224.0
$ card6          <str> 'debit', 'credit', 'debit', 'credit', 'debit'
$ addr1          <f64> 299.0, 204.0, null, 264.0, 315.0
$ addr2          <f64> 87.0, 87.0, null, 87.0, 87.0
$ dist1          <f64> null, null, null, 9.0, null
$ dist2          <f64> null, null, 73.0, null, null
$ P_emaildomain  <str> 'yahoo.com', 'embarqmail.com', 'anonymous.com', 'yahoo.com', null
$ R_emaildomain  <str> null, null, 'anonymou

In [9]:
print(df_train.describe())

shape: (9, 435)
┌────────────┬─────────────┬──────────┬────────────┬───┬────────┬────────┬────────────┬────────────┐
│ statistic  ┆ Transaction ┆ isFraud  ┆ Transactio ┆ … ┆ id_37  ┆ id_38  ┆ DeviceType ┆ DeviceInfo │
│ ---        ┆ ID          ┆ ---      ┆ nDT        ┆   ┆ ---    ┆ ---    ┆ ---        ┆ ---        │
│ str        ┆ ---         ┆ f64      ┆ ---        ┆   ┆ str    ┆ str    ┆ str        ┆ str        │
│            ┆ f64         ┆          ┆ f64        ┆   ┆        ┆        ┆            ┆            │
╞════════════╪═════════════╪══════════╪════════════╪═══╪════════╪════════╪════════════╪════════════╡
│ count      ┆ 590540.0    ┆ 590540.0 ┆ 590540.0   ┆ … ┆ 140985 ┆ 140985 ┆ 140810     ┆ 118666     │
│ null_count ┆ 0.0         ┆ 0.0      ┆ 0.0        ┆ … ┆ 449555 ┆ 449555 ┆ 449730     ┆ 471874     │
│ mean       ┆ 3282269.5   ┆ 0.03499  ┆ 7.3723e6   ┆ … ┆ null   ┆ null   ┆ null       ┆ null       │
│ std        ┆ 170474.3583 ┆ 0.183755 ┆ 4.6172e6   ┆ … ┆ null   ┆ null   ┆ 

In [10]:
df_train = df_train.with_columns(cs.string().fill_null("unknown"))
df_test = df_test.with_columns(cs.string().fill_null("unknown"))

df_test = df_test.rename(
    {col: col.replace("-", "_") for col in df_test.collect_schema().names() if col.startswith("id-")
     })

low_card_cols = [
    "ProductCD", "card4", "card6",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", "id_29",
    "id_34", "id_35", "id_36", "id_37", "id_38", "DeviceType"
]

for col in low_card_cols:
    # Collect all possible categories from both sets
    all_cats = (
        pl.concat([
            df_train.select(col).collect(),
            df_test.select(col).collect()
        ])
        .get_column(col)
        .unique()
        .sort()
        .to_list()
    )
    enum_type = pl.Enum(all_cats)
    df_train = df_train.with_columns(pl.col(col).cast(enum_type))
    df_test = df_test.with_columns(pl.col(col).cast(enum_type))

In [11]:
df_train = df_train.drop("TransactionID")
df_test = df_test.drop("TransactionID")

df_train.sink_parquet("../Data/Processed/train_ready.parquet")
df_test.sink_parquet("../Data/Processed/test_ready.parquet")

In [43]:
df = pl.read_parquet("../Data/Processed/train_ready.parquet")
print(df.head(5).glimpse())


Rows: 5
Columns: 447
$ isFraud                  <u8> 0, 0, 0, 0, 0
$ TransactionDT           <u32> 89341, 89444, 89600, 89844, 89916
$ TransactionAmt          <f32> 306.4700012207031, 200.0, 5.98199987411499, 117.0, 100.0
$ ProductCD              <enum> W, H, C, W, W
$ card1                   <str> '16132', '10047', '3154', '2560', '10057'
$ card2                   <str> '111.0', '111.0', '408.0', '562.0', '225.0'
$ card3                   <str> '150.0', '150.0', '185.0', '150.0', '150.0'
$ card4                  <enum> visa, visa, mastercard, visa, mastercard
$ card5                   <str> '226.0', '226.0', '224.0', '162.0', '224.0'
$ card6                  <enum> debit, credit, debit, credit, debit
$ addr1                   <str> '299.0', '204.0', null, '264.0', '315.0'
$ addr2                   <str> '87.0', '87.0', null, '87.0', '87.0'
$ dist1                   <f64> null, null, null, 9.0, null
$ dist2                   <f64> null, null, 73.0, null, null
$ P_emaildomain           

In [7]:
print(df.select("TransactionDT").tail(3))
print(df.filter(pl.col("P_emaildomain") == pl.col("R_emaildomain" )).select("P_emaildomain","R_emaildomain").head(5) )

shape: (3, 1)
┌───────────────┐
│ TransactionDT │
│ ---           │
│ u32           │
╞═══════════════╡
│ 15451914      │
│ 15452799      │
│ 15453129      │
└───────────────┘
shape: (5, 2)
┌───────────────┬───────────────┐
│ P_emaildomain ┆ R_emaildomain │
│ ---           ┆ ---           │
│ str           ┆ str           │
╞═══════════════╪═══════════════╡
│ anonymous.com ┆ anonymous.com │
│ unknown       ┆ unknown       │
│ anonymous.com ┆ anonymous.com │
│ live.com.mx   ┆ live.com.mx   │
│ unknown       ┆ unknown       │
└───────────────┴───────────────┘


In [38]:
# some feature engineering to help the model make sense of the data

# time features
df = df.with_columns(
    hour_of_the_day = (pl.col("TransactionDT") // 3600 ) % 24,
    day_of_the_week = (pl.col("TransactionDT") // 86400 ) % 7,
    absolute_week = pl.col("TransactionDT") // 604800
)

# P_emaildomain == R_emaildomain ??
df = df.with_columns(
    email_match_status = pl.when(
        (pl.col("P_emaildomain") == "unknown") & (pl.col("R_emaildomain") == "unknown")
    ).then(pl.lit("Both_Unknown"))
    .when(
        (pl.col("P_emaildomain") == "unknown") | (pl.col("R_emaildomain") == "unknown")
    ).then(pl.lit("Partial_Unknown"))
    .when(
        (pl.col("P_emaildomain") == pl.col("R_emaildomain")) & (pl.col("P_emaildomain") != "unknown")
    ).then(pl.lit("Exact_Match"))
    .otherwise(pl.lit("Explicit_Mismatch"))

    )

# uid_Count  -> uid  = card1 + addr1 + P_emaildomain 
# The "Good Customer" Metric (How often does this exact identity buy?)
df = (df.with_columns(
        uid_string = pl.concat_str(["card1", "addr1", "P_emaildomain"], separator="_", ignore_nulls=True)
    )
    .with_columns( uid_c = pl.len().over("uid_string")
                  
    )

    )

# UID Diversity per Card 
# The "Stolen Card" Metric (How many different identities use this card?)
df = df.with_columns(
    uid_diversity = pl.col("uid_string").n_unique().over("card1")
)

# transaction ratio to card mean (flags any potential micro fraud or massive cash out fraud )

df = df.with_columns(
    amt_ratio_to_card_mean = pl.col("TransactionAmt") / pl.col("TransactionAmt").mean().over("card1")
)

# how many unique devices per card

df = df.with_columns(
    device_diversity = pl.col("DeviceInfo").n_unique().over("card1")
)


df = (df.with_columns(
    absolute_hour = pl.col("TransactionDT") // 3600,
    absolute_day = pl.col("TransactionDT") // 86400
).with_columns(
    # How many times did THIS specific card swipe during THIS specific hour?
    hourly_tx_count = pl.col("TransactionAmt").len().over(["absolute_hour","card1"]) ,
    # How much total money did this card spend in this hour?
    hourly_tx_sum = pl.col("TransactionAmt").sum().over(["absolute_hour","card1"]),

    # same logic with different time windows
    daily_tx_count = pl.len().over(["absolute_day", "card1"]),
    daily_tx_sum = pl.col("TransactionAmt").sum().over(["absolute_day", "card1"]),

    weekly_tx_count = pl.len().over(["absolute_week", "card1"]),
    weekly_tx_sum = pl.col("TransactionAmt").sum().over(["absolute_week", "card1"])
)
)

df = df.with_columns(
    transaction_cents = pl.col("TransactionAmt") % 1
)

In [54]:
df = df.sort("TransactionDT")
        
# 1. Base Helpers & Cyclical Time
df = df.with_columns(
    hour_of_the_day = ((pl.col("TransactionDT") // 3600) % 24).cast(pl.UInt32),
    day_of_the_week = ((pl.col("TransactionDT") // 86400) % 7).cast(pl.UInt32),
    absolute_hour = (pl.col("TransactionDT") // 3600).cast(pl.UInt32),
    absolute_day = (pl.col("TransactionDT") // 86400).cast(pl.UInt32),
    absolute_week = (pl.col("TransactionDT") // 604800).cast(pl.UInt32)
)

# 2. String Logic (Email Match Status)
df = df.with_columns(
    email_match_status = (
        pl.when(
            (pl.col("P_emaildomain") == "unknown") & (pl.col("R_emaildomain") == "unknown")
        ).then(pl.lit("Both_Unknown"))
        .when(
            (pl.col("P_emaildomain") == "unknown") | (pl.col("R_emaildomain") == "unknown")
        ).then(pl.lit("Partial_Unknown"))
        .when(
            (pl.col("P_emaildomain") == pl.col("R_emaildomain")) & (pl.col("P_emaildomain") != "unknown")
        ).then(pl.lit("Exact_Match"))
        .otherwise(pl.lit("Explicit_Mismatch"))
    ).cast(pl.Utf8)
)

# 3. Core Feature Engineering Chain
df = (
    df.with_columns(
        uid_string = pl.concat_str(["card1", "addr1", "P_emaildomain"], separator="_", ignore_nulls=True),
        transaction_cents = (pl.col("TransactionAmt") % 1).cast(pl.Float32)
    )
    .with_columns(
        
        # --- PLUGGING THE TIME LEAKS (Cumulative instead of Global) ---
        
        # How many times has this uid appeared UP TO THIS ROW?
        uid_c = pl.lit(1).cum_sum().over("uid_string").cast(pl.UInt32),
        
        # How many unique UIDs has this card used UP TO THIS ROW?
        uid_diversity = pl.col("uid_string").is_first_distinct().cum_sum().over("card1").cast(pl.UInt32),
        
        # How many unique devices has this card used UP TO THIS ROW?
        device_diversity = pl.col("DeviceInfo").is_first_distinct().cum_sum().over("card1").cast(pl.UInt32),
        
        # Ratio to cumulative mean (What is their average spend UP TO THIS ROW?)
        amt_ratio_to_card_mean = (
            pl.col("TransactionAmt") / (
                pl.col("TransactionAmt").cum_sum().over("card1") / 
                pl.col("TransactionAmt").cum_count().over("card1")
            )
        ).cast(pl.Float32),
        
        # --- TIME-SAFE WINDOWS (These are safe because absolute_hour isolates them) ---
        hourly_tx_count = pl.len().over(["absolute_hour", "card1"]).cast(pl.UInt32),
        hourly_tx_sum = pl.col("TransactionAmt").sum().over(["absolute_hour", "card1"]).cast(pl.Float32),
        
        daily_tx_count = pl.len().over(["absolute_day", "card1"]).cast(pl.UInt32),
        daily_tx_sum = pl.col("TransactionAmt").sum().over(["absolute_day", "card1"]).cast(pl.Float32),
        
        weekly_tx_count = pl.len().over(["absolute_week", "card1"]).cast(pl.UInt32),
        weekly_tx_sum = pl.col("TransactionAmt").sum().over(["absolute_week", "card1"]).cast(pl.Float32)
    )
)

# 4. Clean up temporary helpers
df = df.drop([
     
    "absolute_hour", 
    "absolute_day", 
    "absolute_week"
])

In [55]:
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(1000)
print(
    df.filter(pl.col("isFraud") == 1).select(
        "TransactionDT",
        "TransactionAmt",
        "transaction_cents",
        "hourly_tx_sum", 
        "hourly_tx_count", 
        "daily_tx_count",
        "daily_tx_sum",
        "weekly_tx_count",
        "weekly_tx_sum",
        "email_match_status", 
        "uid_string", 
        "uid_c", 
        "uid_diversity",
        "amt_ratio_to_card_mean",
        "device_diversity",
        "isFraud"
    ).head(100)
)

shape: (100, 16)
┌───────────────┬────────────────┬───────────────────┬───────────────┬─────────────────┬────────────────┬──────────────┬─────────────────┬───────────────┬────────────────────┬───────────────────────┬───────┬───────────────┬────────────────────────┬──────────────────┬─────────┐
│ TransactionDT ┆ TransactionAmt ┆ transaction_cents ┆ hourly_tx_sum ┆ hourly_tx_count ┆ daily_tx_count ┆ daily_tx_sum ┆ weekly_tx_count ┆ weekly_tx_sum ┆ email_match_status ┆ uid_string            ┆ uid_c ┆ uid_diversity ┆ amt_ratio_to_card_mean ┆ device_diversity ┆ isFraud │
│ ---           ┆ ---            ┆ ---               ┆ ---           ┆ ---             ┆ ---            ┆ ---          ┆ ---             ┆ ---           ┆ ---                ┆ ---                   ┆ ---   ┆ ---           ┆ ---                    ┆ ---              ┆ ---     │
│ u32           ┆ f32            ┆ f32               ┆ f32           ┆ u32             ┆ u32            ┆ f32          ┆ u32             ┆ f32       